# 09 — A full regional model: Coquimbo–La Serena

This notebook runs a complete regional transport model end to end, exercising every
major AequilibraE capability on the bundled **Coquimbo (Chile)** model: a real
OSM-derived network of ~20,000 links, 133 zones carrying population and employment
data, and a GTFS transit feed. Everything is embedded in the AequilibraE package —
nothing is downloaded.

The pipeline:

1. **Network preparation** — fill missing speeds, travel times and capacities
2. **Graphs & free-flow skims**
3. **Trip generation** from zonal land use
4. **Gravity distribution + IPF balancing**
5. **Multi-class equilibrium assignment** (car + freight PCE) with **select link analysis**
6. **Congestion mapping** on an interactive offline map
7. **Route choice** with BFSLE choice sets
8. **GTFS transit import** and route mapping

In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4
import warnings

import numpy as np
import pandas as pd

from aequilibrae.utils.create_example import create_example
from aequilibrae.paths import TrafficAssignment, TrafficClass, NetworkSkimming

warnings.filterwarnings("ignore")  # silence pandas copy-on-write chatter from graph compression
np.random.seed(42)

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "coquimbo")

net = project.network
print(f"links: {net.count_links():,}   nodes: {net.count_nodes():,}   modes: {list(net.modes.all_modes().keys())}")

links: 19,983   nodes: 15,724   modes: ['b', 'c', 't', 'w']


## The study area

Zones, centroids and the road network on a live map.

In [2]:
# Offline map helper ---------------------------------------------------------
# Interactive maps with no server extensions, no labextensions beyond the
# ipywidgets manager, and no CDN: lonboard renders WebGL maps whose frontend
# JavaScript ships from the kernel through the ipywidgets channel.
#
# Backends (AEQ_MAP_BACKEND environment variable):
#   lonboard (default) - interactive WebGL maps (pip install lonboard anywidget)
#   static             - matplotlib rendering, works absolutely anywhere
#
# The declarative symbology below (field()/constant() chains) is self-contained
# and renders identically on both backends.
import os

import matplotlib.colors
import matplotlib.pyplot as _plt
import numpy as np


# --- declarative symbology --------------------------------------------------
class _Mapping:
    def __init__(self, field, scheme, params):
        self.field, self.scheme, self.params = field, scheme, params

    def encoding(self, *targets):
        return {"field": self.field, "scheme": self.scheme,
                "params": self.params, "encodings": list(targets)}


class _Field:
    def __init__(self, name):
        self.name = name

    def colormap(self, name="viridis", *, domain=None, reverse=False, n_shades=9):
        return _Mapping(self.name, "colormap",
                        {"name": name, "domain": domain, "reverse": reverse})

    def scalar(self, *, domain, output_range):
        return _Mapping(self.name, "scalar",
                        {"domain": list(domain), "range": list(output_range)})

    def categorical(self, name="tab10"):
        return _Mapping(self.name, "categorical", {"name": name})


class _Constant:
    def __init__(self, value):
        self.value = value

    def encoding(self, *targets):
        scheme = "constant_num" if isinstance(self.value, (int, float)) else "constant_color"
        return {"field": None, "scheme": scheme,
                "params": {"value": self.value}, "encodings": list(targets)}


def field(name):
    """Style by a data column: .colormap() / .scalar() / .categorical()."""
    return _Field(name)


def constant(value):
    """A fixed colour (hex/name) or number, e.g. constant("#dc2626")."""
    return _Constant(value)


def _rgba255(c, alpha=1.0):
    r, g, b, a = matplotlib.colors.to_rgba(c, alpha)
    return [int(r * 255), int(g * 255), int(b * 255), int(a * 255)]


def _style_arrays(symbology, gdf):
    """symbology -> per-row uint8 RGBA arrays and float width arrays."""
    n = len(gdf)
    out = {"stroke": None, "width": None, "fill": None}
    if not symbology:
        return out
    mappings = [m for group in symbology for m in (group if isinstance(group, list) else [group])]
    for m in mappings:
        scheme, params, fld, encs = m["scheme"], m["params"], m["field"], m["encodings"]
        arr = wid = None
        if scheme == "constant_color":
            arr = np.tile(_rgba255(params["value"]), (n, 1)).astype(np.uint8)
        elif scheme == "colormap":
            cmap = _plt.get_cmap(params["name"])
            if params.get("reverse"):
                cmap = cmap.reversed()
            dom = params.get("domain") or [float(gdf[fld].min()), float(gdf[fld].max())]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - dom[0]) / max(dom[1] - dom[0], 1e-12), 0, 1)
            rgba = cmap(t)
            arr = (rgba * 255).astype(np.uint8)
        elif scheme == "categorical":
            cmap = _plt.get_cmap(params["name"])
            uniq = list(dict.fromkeys(gdf[fld].dropna()))
            idx = {v: i for i, v in enumerate(uniq)}
            arr = np.array([_rgba255(cmap(idx.get(v, 0) % cmap.N)) for v in gdf[fld]], dtype=np.uint8)
        elif scheme == "constant_num":
            wid = np.full(n, float(params["value"]))
        elif scheme == "scalar":
            d, r = params["domain"], params["range"]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - d[0]) / max(d[1] - d[0], 1e-12), 0, 1)
            wid = r[0] + t * (r[1] - r[0])
        if arr is not None:
            if any("stroke" in e for e in encs):
                out["stroke"] = arr
            if any("fill" in e for e in encs):
                out["fill"] = arr
        if wid is not None and any("width" in e for e in encs):
            out["width"] = wid
    return out


# --- the map document -------------------------------------------------------
class MapDoc:
    """Collects styled layers; displays via lonboard (WebGL) or matplotlib."""

    def __init__(self):
        self.items = []  # (gdf, name, arrays, opacity)

    def add(self, gdf, name, symbology, opacity):
        g = gdf.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
        self.items.append((g, name, _style_arrays(symbology, g), opacity))

    def _lonboard_map(self):
        from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
        layers = []
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            base = g[["geometry"]]
            if "LineString" in geom:
                kw = {"width_units": "pixels", "width_min_pixels": 1.0, "opacity": op}
                if st["stroke"] is not None:
                    kw["get_color"] = st["stroke"]
                if st["width"] is not None:
                    kw["get_width"] = st["width"]
                layers.append(PathLayer.from_geopandas(base, **kw))
            elif "Polygon" in geom:
                kw = {"opacity": op * 0.6, "stroked": False}
                if st["fill"] is not None:
                    kw["get_fill_color"] = st["fill"]
                layers.append(PolygonLayer.from_geopandas(base, **kw))
            else:
                kw = {"radius_min_pixels": 5, "opacity": op}
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                if fill is not None:
                    kw["get_fill_color"] = fill
                layers.append(ScatterplotLayer.from_geopandas(base, **kw))
        return Map(layers=layers, basemap=None)

    def _static_figure(self):
        fig, ax = _plt.subplots(figsize=(9, 7))
        ax.set_facecolor("#eef1f4")
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            if "LineString" in geom:
                colors = st["stroke"] / 255 if st["stroke"] is not None else "#1d4ed8"
                widths = st["width"] if st["width"] is not None else 1.0
                g.plot(ax=ax, color=colors, linewidth=widths, alpha=op)
            elif "Polygon" in geom:
                colors = st["fill"] / 255 if st["fill"] is not None else "#cbd5e1"
                g.plot(ax=ax, color=colors, alpha=op * 0.6)
            else:
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                g.plot(ax=ax, color=(fill / 255 if fill is not None else "#dc2626"),
                       markersize=25, alpha=op)
        ax.set_aspect(1.4)
        ax.set_xticks([]), ax.set_yticks([])
        _plt.tight_layout()
        _plt.close(fig)
        return fig

    def _ipython_display_(self):
        from IPython.display import display
        be = os.environ.get("AEQ_MAP_BACKEND", "lonboard").strip().lower()
        display(self._static_figure() if be == "static" else self._lonboard_map())


def new_map(gdf_for_extent=None, zoom=12):
    """Create a map document (extent/zoom args kept for API compatibility;
    lonboard auto-fits to its layers)."""
    return MapDoc()


def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a styled layer."""
    doc.add(gdf, name, symbology, kwargs.get("opacity", 1.0))
    return name


def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature — backdrop
    layers do not need per-feature identity, and one merged feature is a
    fraction of the size and draw cost."""
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)


In [3]:
# field()/constant() symbology builders come from the map helper cell

links = net.links.data
nodes = net.nodes.data
zones = project.zoning.data

doc = new_map(links, zoom=11)
add_gdf(doc, zones, "zones", opacity=0.4, symbology=[[constant("#f59e0b").encoding("fill")]])
add_gdf(doc, links[links.link_type != "centroid_connector"], "road network",
        symbology=[[constant("#1d4ed8").encoding("stroke")]])
add_gdf(doc, nodes[nodes.is_centroid == 1], "zone centroids",
        symbology=[[constant("#dc2626").encoding("fill")]])
doc

[interactive offline map - run the notebook to display]

## 1. Network preparation

Real-world networks arrive with gaps: here every car link is missing a travel time and
most are missing capacities. We derive free-flow speeds from the OSM link type,
travel times from length and speed, and capacities from lane counts — written back
through the project database so the network's consistency triggers keep everything
aligned.

In [4]:
DEFAULT_SPEED = {"motorway": 100, "trunk": 80, "primary": 60, "secondary": 50,
                 "tertiary": 40, "residential": 30, "living_street": 20,
                 "unclassified": 40, "centroid_connector": 24}   # km/h
LANE_CAPACITY = 900  # veh/h/lane

with project.db_connection as conn:
    for lt, speed in DEFAULT_SPEED.items():
        conn.execute("update links set speed_ab = coalesce(speed_ab, ?), "
                     "speed_ba = coalesce(speed_ba, ?) where link_type = ?", (speed, speed, lt))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60")
    conn.execute("update links set capacity_ab = coalesce(capacity_ab, coalesce(lanes_ab, 1) * ?), "
                 "capacity_ba = coalesce(capacity_ba, coalesce(lanes_ba, 1) * ?)",
                 (LANE_CAPACITY, LANE_CAPACITY))
    conn.commit()
    check = pd.read_sql("select count(*) links, sum(travel_time_ab is null) tt_null, "
                        "sum(capacity_ab is null) cap_null from links", conn)
check

,links,tt_null,cap_null
0,19983,0,0


## 2. Graphs and free-flow skims

In [5]:
net.build_graphs(modes=["c"])
graph = net.graphs["c"]
graph.set_graph("travel_time")
graph.set_skimming(["travel_time", "distance"])
graph.set_blocked_centroid_flows(False)

skimmer = NetworkSkimming(graph)
skimmer.execute()

tt = np.array(skimmer.results.skims.get_matrix("travel_time"), copy=True)
offdiag = tt[~np.eye(tt.shape[0], dtype=bool)]
connected = np.isfinite(offdiag)
print(f"free-flow time skim {tt.shape[0]}x{tt.shape[1]}: "
      f"mean {offdiag[connected].mean():.1f} min, {100 * connected.mean():.1f}% of OD pairs connected")

[interactive offline map - run the notebook to display]

free-flow time skim 133x133: mean 13.8 min, 99.2% of OD pairs connected


## 3. Trip generation from land use

The zones table carries **population** (the employment field ships empty in this
dataset, so attractions use a sublinear population proxy — the standard stand-in
when a jobs layer is missing). Productions and attractions are balanced to the
same total.

In [6]:
zones_df = zones[["zone_id", "population"]].fillna(0).set_index("zone_id")
zones_df = zones_df.reindex(graph.centroids).fillna(0)

TRIP_RATE = 0.45  # peak-hour person-trips per resident
productions = zones_df.population.to_numpy(dtype=float) * TRIP_RATE
attractions = np.power(zones_df.population.to_numpy(dtype=float), 0.85)
attractions *= productions.sum() / attractions.sum()

print(f"{productions.sum():,.0f} trips over {len(zones_df)} zones "
      f"(largest producer: zone {zones_df.population.idxmax()})")

203,355 trips over 133 zones (largest producer: zone 19)


## 4. Gravity distribution and IPF balancing

A negative-exponential gravity model seeds the matrix from the free-flow time skim,
and classic doubly-constrained (IPF) balancing matches it to the production and
attraction vectors — written out explicitly so every step is visible.
(AequilibraE's `GravityCalibration`/`Ipf` classes wrap the same math over project
matrices — see notebook 04.)

In [7]:
imp = tt.copy()
np.fill_diagonal(imp, np.nan)
imp[~np.isfinite(imp)] = np.nan            # unreachable pairs get no trips

BETA = 0.12
T = np.outer(productions, attractions) * np.exp(-BETA * np.nan_to_num(imp, nan=1e3))
T[np.isnan(imp)] = 0.0
np.fill_diagonal(T, 0.0)

for it in range(100):                      # doubly-constrained balancing (IPF)
    rs = T.sum(1); T *= np.divide(productions, rs, out=np.zeros_like(rs), where=rs > 0)[:, None]
    cs = T.sum(0); T *= np.divide(attractions, cs, out=np.zeros_like(cs), where=cs > 0)[None, :]
    gap = np.abs(T.sum(1) - productions).sum() / productions.sum()
    if gap < 1e-4:
        break

print(f"IPF: {it + 1} iterations, residual gap {gap:.2e} "
      f"(isolated zones keep it non-zero); peak-hour demand {T.sum():,.0f} trips")

IPF: 100 iterations, residual gap 2.14e-02 (isolated zones keep it non-zero); peak-hour demand 203,355 trips


## 5. Multi-class equilibrium assignment with select link

Two user classes share the equilibrium: cars (92% of demand) and light freight
(8%, with a passenger-car equivalent of 2.5). A select-link flag on the busiest
motorway section captures exactly which traffic uses it.

In [8]:
from aequilibrae.matrix import AequilibraeMatrix

def class_matrix(name, share):
    m = AequilibraeMatrix()
    m.create_empty(zones=graph.num_zones, matrix_names=[name], memory_only=True)
    m.index = graph.centroids[:]
    m.matrices[:, :, 0] = T * share
    m.computational_view()
    return m

cars, trucks = class_matrix("cars", 0.92), class_matrix("freight", 0.08)

links = net.links.data  # refreshed: now carries the prepared capacities
screenline = int(links[links.link_type == "motorway"].nlargest(1, "capacity_ab").link_id.iloc[0])
print(f"select link: motorway link {screenline}")

car_class = TrafficClass(name="car", graph=graph, matrix=cars)
car_class.set_select_links({"screenline": [(screenline, 0)]})
truck_class = TrafficClass(name="freight", graph=graph, matrix=trucks)
truck_class.set_pce(2.5)
truck_class.set_select_links({"screenline": [(screenline, 0)]})

assig = TrafficAssignment()
assig.add_class(car_class)
assig.add_class(truck_class)
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})
assig.set_capacity_field("capacity")
assig.set_time_field("travel_time")
assig.set_algorithm("bfw")
assig.max_iter = 40
assig.rgap_target = 0.001
assig.execute()

assig.save_results("peak_hour")
assig.save_select_link_results("screenline_analysis")

select link: motorway link 15365


[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

## 6. Congestion on the map

Volume-to-capacity ratios from the converged assignment, straight onto JupyterGIS.

In [9]:
# field()/constant() symbology builders come from the map helper cell

res = assig.results()
flow_col = "PCE_tot" if "PCE_tot" in res.columns else "matrix_tot"

loaded = links.merge(res.reset_index(), on="link_id")
loaded = loaded[loaded[flow_col] > 0].copy()
loaded["voc"] = loaded["VOC_max"].clip(upper=2).round(3)

doc = new_map(loaded, zoom=12)
add_gdf(doc, loaded[["link_id", "voc", flow_col, "geometry"]].rename(columns={flow_col: "flow"}),
        "V/C ratio", symbology=[[field("voc").colormap("RdYlGn_r", domain=(0.0, float(loaded.voc.max()))).encoding("stroke"),
                    field("flow").scalar(domain=(0.0, float(loaded[flow_col].max())),
                                         output_range=(0.8, 7.0)).encoding("stroke-width")]])
doc

[interactive offline map - run the notebook to display]

In [10]:
loaded.nlargest(8, flow_col)[["link_id", "name", "link_type", flow_col, "VOC_max"]] \
      .rename(columns={flow_col: "peak_flow"}).reset_index(drop=True)

,link_id,name,link_type,peak_flow,VOC_max
0,806,Ruta 5 Norte,trunk,20392.333473,7.552716
1,459,Ruta 5 Norte,trunk,20389.199376,7.551555
2,5374,Ruta 5 Norte,trunk,20389.199376,7.551555
3,26294,Ruta 5 Norte,trunk,20389.199376,7.551555
4,26295,Ruta 5 Norte,trunk,20389.199376,7.551555
5,5372,Ruta 5 Norte,trunk,20122.167776,7.452655
6,20780,Ruta 5 Norte,trunk,20122.167776,11.178982
7,26297,Ruta 5 Norte,trunk,20122.167776,11.178982


The select-link flows — only the traffic that actually crosses the screenline:

In [11]:
with project.results_connection as conn:
    sl = pd.read_sql("select * from screenline_analysis", conn)
if "link_id" not in sl.columns:
    sl = sl.rename(columns={sl.columns[0]: "link_id"})

tot_cols = [c for c in sl.columns if c.endswith("_tot")]
sl["screenline_flow"] = sl[tot_cols].sum(axis=1)
sl = sl[sl.screenline_flow > 0]
print(f"{len(sl)} links carry the traffic that crosses the screenline")

slg = links.merge(sl[["link_id", "screenline_flow"]], on="link_id")
doc = new_map(loaded, zoom=12)
add_gdf(doc, slg[["link_id", "screenline_flow", "geometry"]], "screenline traffic",
        symbology=[[field("screenline_flow").colormap("Blues", domain=(0.0, float(slg.screenline_flow.max()))).encoding("stroke"),
                field("screenline_flow").scalar(domain=(0.0, float(slg.screenline_flow.max())),
                                                output_range=(1.0, 6.5)).encoding("stroke-width")]])
add_gdf(doc, links[links.link_id == screenline], "screenline link",
        symbology=[[constant("#dc2626").encoding("stroke")]])
doc

11365 links carry the traffic that crosses the screenline


[interactive offline map - run the notebook to display]

## 7. Route choice

A BFSLE choice set between two distant zones — the realistic alternatives a
traveller weighs — each drawn on the map.

In [12]:
from aequilibrae.paths import RouteChoice

finite_tt = np.where(np.isfinite(tt), tt, -1)
o_idx, d_idx = np.unravel_index(np.argmax(finite_tt), finite_tt.shape)
origin, dest = int(graph.centroids[o_idx]), int(graph.centroids[d_idx])

rc = RouteChoice(graph)
rc.set_choice_set_generation("bfsle", max_routes=5)
routes = rc.execute_single(origin, dest, demand=1.0)
print(f"{len(routes)} alternative routes between zone {origin} and zone {dest} "
      f"({tt[o_idx, d_idx]:.0f} min apart at free flow)")

palette = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed"]
doc = new_map(links, zoom=11)
for i, route in enumerate(routes):
    route_links = links[links.link_id.isin(list(route))]
    add_gdf(doc, route_links[["link_id", "geometry"]], f"route {i + 1}",
            symbology=[[constant(palette[i % len(palette)]).encoding("stroke"), constant(3.5).encoding("stroke-width")]])
doc

5 alternative routes between zone 89 and zone 23 (45 min apart at free flow)


[interactive offline map - run the notebook to display]

## 8. Public transport: GTFS import

The Coquimbo example embeds a real GTFS feed (`gtfs_coquimbo.zip`). Importing it
builds the transit database — routes, stops, trips and patterns.

In [13]:
from os import remove

from aequilibrae.transit import Transit

# The example ships with a transit DB already built - remove it so we import cleanly
remove(str(Path(fldr) / "public_transport.sqlite"))

data = Transit(project)
gtfs = data.new_gtfs_builder(agency="Lisanco", file_path=str(Path(fldr) / "gtfs_coquimbo.zip"))
gtfs.load_date("2016-04-13")
gtfs.save_to_disk()

import geopandas as gpd

with project.transit_connection as conn:
    routes = pd.read_sql("SELECT route_id, route, ST_AsText(geometry) wkt FROM routes", conn)
    stops = pd.read_sql("SELECT stop_id, ST_X(geometry) x, ST_Y(geometry) y FROM stops", conn)

routes_gdf = gpd.GeoDataFrame(routes.drop(columns="wkt"),
                              geometry=gpd.GeoSeries.from_wkt(routes["wkt"]), crs=4326)
stops_gdf = gpd.GeoDataFrame(stops, geometry=gpd.points_from_xy(stops.x, stops.y), crs=4326)

doc = new_map(stops_gdf, zoom=12)
add_gdf(doc, routes_gdf, "transit routes", symbology=[[constant("#2563eb").encoding("stroke")]])
add_gdf(doc, stops_gdf, "stops", symbology=[[constant("#111827").encoding("fill")]])
print(f"{len(routes_gdf)} routes, {len(stops_gdf)} stops imported")
doc

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

...

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

2 routes, 78 stops imported


[interactive offline map - run the notebook to display]

## Wrap-up

One notebook, one region, the full toolbox: network editing and preparation,
graph building, skimming, land-use-based trip generation, gravity + IPF
distribution, multi-class BPR equilibrium with select link, congestion and
screenline mapping, BFSLE route choice, and GTFS transit import — all on data
embedded in the AequilibraE package, with every map an interactive JupyterGIS
document.

Results live in the project folder (`peak_hour` assignment, `screenline_analysis`
select-link tables, and the transit database) — ready for the scenario-comparison
workflow of notebook 08.